# Introduction to the HHO module

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

The module `HHO_kernel` in this repository provides an implementation of the HHO solver of the Poisson equation
$$
-u'' = f
$$
on an interval of $\mathbb{R}$.
The class implementing this solver is called `HHO_poisson`, so let us start and import it right away.

In [ ]:
from HHO_kernel import HHO_poisson

## Setting up the solver

The following information is needed to initialize the solver:
- The grid for the discretization.
- The degree $k$ of the polynomial unknowns on the cells (it defaults to the lowest-order realization $k=0$).
- The basis used to represent the polynomial space on the cells.
    - We have implemented a monomial basis (which is the default), applied by the keyword argument `basis="monomial"` at initialization.\
    Note, however, that the monomial basis becomes numerically unstable for higher-degree polynomial spaces, and should be orthogonalized.
    - We have also made available the basis of Legendre polynomials, implemented via the `scipy.special` module. Its current performance is much slower than the monomial basis. It is activated through the keyword argument `basis="legendre"` at initialization.

Let's say we want to discretize the Poisson equation on the domain $(0, \pi)$ on a grid consisting of 9 cells, with degree $k=2$ for the HHO cell unknowns and using the monomial basis. This is done as follows.

In [ ]:
x_left = 0.0
x_right = np.pi
xx = np.linspace(x_left, x_right, 10)
poisson = HHO_poisson(xx, degree=2, basis="monomial")

Having built the solver object, we can now specify the boundary conditions for our problem. This is done by first setting the *type* of boundary conditions applied. This is done through setting the `boundary_conditions` attribute of the `poisson` solver. It must be of the form `'XX'` with each `X` being either `D` (for Dirichlet) or `N` (for Neumann). The first `X` sets the boundary condition on the left of the interval, and the second `X` on the right.

Dirichlet boundary conditions can be inhomogeneous, but the solver only takes homogeneous Neumann conditions.

Preparing the solver for a problem with a homogeneous Neumann boundary condition on the left and a Dirichlet condition on the right is now done as follows.

In [ ]:
poisson.boundary_conditions = 'ND'

## Solving the Poisson equation

A boundary-value can now be solved by calling the `poisson.solve()` method. It takes the following arguments:
* The function $f$ on the right-hand side of the Poisson equation (operating on `numpy` arrays).
* The values of the boundary conditions as optional keyword arguments:
    * `bc_left` provides the value of the Dirichlet condition on the left when `boundary conditions` is of the form `'DX'`.
    * `bc_right` provides the value of the Dirichlet condition on the right when `boundary conditions` is of the form `'XD'`.
    * When `boundary_conditions` is set to `NN`, the average must be provided to obtain a unique solution, which is done through the keyword argument `average`.

We illustrate this for the case when $u(x) = \cos(8x) + x^2$, so $f(x) = 64 \cos(8x) - 2$. The solution indeed satisfies a homogeneous Neumann condition on the left. The Poisson equation can now be solved.

In [ ]:
u = lambda x: np.cos(8*x) + x**2
f = lambda x: 64*np.cos(8*x) - 2
poisson.solve(f, bc_right=u(x_right))

`solve` computes the degrees of freedom of the HHO method for the approximation of $u$, which consist of values at the vertices of the mesh (faces in the case of a higher-dimensional problem) and the values of $L^2$ projections on each cell. These values are now stored in the `poisson` solver. 
The values at the faces are stored in the `solution_face` attribute of the solver.

In [ ]:
print(f'DOFs of the approximation at the faces: {poisson.solution_face}')

Each cell is represented in the solver by an element of the `cells` attribute, and the solution on cell `i` (consisting of 3 numbers when the cell unknowns are polynomials of degree $k=2$) are stored in `poisson.cells[i].solution`.

In [ ]:
for i, cell in enumerate(poisson.cells):
    print(f'DOFs on cell {i}: {cell.solution}')

## Visualizing the solution

The solver comes equipped with a method `plot()` to visualize the DOFs of the HHO approximation, as follows, to which we pass:
* A `matplotlib.axes.Axes` object for the plot to be displayed on.
* A number of optional arguments to specify what we want to visualize:
    * "faces" for the DOFs of the HHO approximation at the faces.
    * "cells" for the DOFs of the HHO approximation on the cells.
    * "reconstruction" for the reconstruction of the potential $u$ based on the DOFs that result from the discrete problem that was solved by `poisson.solve()`.

In [ ]:
# Plot of the DOFs of the HHO approximation
xx_plot = np.linspace(x_left, x_right, 100)
fig, axs = plt.subplots(1,2,figsize=(10,4))
poisson.plot(axs[0], "faces", "cells")
axs[0].legend()
# Plot the reconstruction for the potential and compare it with u
axs[1].plot(xx_plot, u(xx_plot), 'black', label="$u(x)$")
poisson.plot(axs[1], "faces", "reconstruction")
axs[1].legend();

We clearly see from the plots that no continuity is imposed on neighboring cell and face unknowns. The reconstruction, on the other hand, equals the face unknowns at all the faces and is thus continuous. This is a particular property of the HHO method in 1D when $k\geq1$.

Show solution with average condition
Show problems for higher order
Show that it is relieved with legendre